In [1]:
import arff
import pandas as pd

# загружаем данные
with open("c:/Personal/Science/Data/Binary_classification/churn.arff", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],
                  columns=[a[0] for a in dataset["attributes"]])
df.head()

,state,account_length,area_code,international_plan,voice_mail_plan,number_vmail_messages,total_day_minutes,total_day_calls,total_day_charge,total_eve_minutes,total_eve_calls,total_eve_charge,total_night_minutes,total_night_calls,total_night_charge,total_intl_minutes,total_intl_calls,total_intl_charge,number_customer_service_calls,class
0,16,128.0,415,0,1,25.0,265.1,110.0,45.07,197.4,99.0,16.78,244.7,91.0,11.01,10.0,3.0,2.70,1,0
1,35,107.0,415,0,1,26.0,161.6,123.0,27.47,195.5,103.0,16.62,254.4,103.0,11.45,13.7,3.0,3.70,1,0
2,31,137.0,415,0,0,0.0,243.4,114.0,41.38,121.2,110.0,10.30,162.6,104.0,7.32,12.2,5.0,3.29,0,0
3,35,84.0,408,1,0,0.0,299.4,71.0,50.90,61.9,88.0,5.26,196.9,89.0,8.86,6.6,7.0,1.78,2,0
4,36,75.0,415,1,0,0.0,166.7,113.0,28.34,148.3,122.0,12.61,186.9,121.0,8.41,10.1,3.0,2.73,3,0


In [5]:
df.dtypes

state                             object
account_length                   float64
area_code                         object
international_plan                object
voice_mail_plan                   object
number_vmail_messages            float64
total_day_minutes                float64
total_day_calls                  float64
total_day_charge                 float64
total_eve_minutes                float64
total_eve_calls                  float64
total_eve_charge                 float64
total_night_minutes              float64
total_night_calls                float64
total_night_charge               float64
total_intl_minutes               float64
total_intl_calls                 float64
total_intl_charge                float64
number_customer_service_calls     object
class                             object
dtype: object

In [5]:
df['class'] = df['class'].astype(float)
df['number_customer_service_calls'] = df['number_customer_service_calls'].astype(float)
df['international_plan'] = df['international_plan'].astype(int)
df['voice_mail_plan'] = df['voice_mail_plan'].astype(int)

In [ ]:
column_types = {
        "sufficient_resources_nominal_columns" : ['state', 'area_code' ],
        "insufficient_resources_nominal_columns": None,
        "binary": None,
        "ordinal": None,
        "inclusion": None,
        "similarity":None,
    }
ANOC_threshold = 100

In [9]:
y = df["class"]
X = df.drop(["class"], axis=1)

In [ ]:
import time
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from preprocessor import clean_dataset
from meta_encoder import MetaEncoder
from sklearn.base import clone

# models
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

#from approach import encode_with_approach

model_types = {
    "LogReg": "ATI",
    "MLP": "ATI",
    "LinearSVM": "ATI",
    "CART": "tree",
    "RF": "tree",
    "CatBoost": "tree",
    "XGBoost": "tree",
    "LightGBM": "tree"
}
target_encoders = ['Leave-one-out', 'K-fold', 'CatBoost', 'CV-GLMM', 'WOE']

nominal_cols = column_types["sufficient_resources_nominal_columns"].copy()
if column_types["insufficient_resources_nominal_columns"]:
    nominal_cols.extend(column_types["insufficient_resources_nominal_columns"])

max_categories = df[nominal_cols].nunique().max()
ANOC = df.shape[0] / max_categories
# ANOC = 101

encode_with_target: bool = False

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():

    # initializing guide parameters
    if model_types[model_name] == 'ATI':
        if ANOC > ANOC_threshold:
            encode_with_target = False
        else:
            encode_with_target = True
    else:  # 'tree-based'
        if ANOC > ANOC_threshold:
            encode_with_target = True
        else:
            encode_with_target = False

    target_encoders_to_apply = []

    if encode_with_target:
        target_encoders_to_apply = target_encoders
    else: # it just a mark that we use target-agnostic encoders
        target_encoders_to_apply = ["OneHot"]

    for nominal_encoder in target_encoders_to_apply:

        # creating parameters for the meta-encoder
        encode_columns_as = {}
        for type, columns in column_types.items():
            if type == "binary":
                encode_columns_as["Binary"] = columns
                continue
            if type == "similarity":
                encode_columns_as["Similarity"] = columns
                continue
            if type == "ordinal":
                encode_columns_as["Ordinal"] = columns
                continue
            if type == "inclusion":
                encode_columns_as["RankHot"] = columns
                continue

        encoder_name: str
        if not encode_with_target:
            if column_types["sufficient_resources_nominal_columns"] :
                encode_columns_as["OneHot"] = column_types["sufficient_resources_nominal_columns"]
                encoder_name = "OneHot"
            if column_types["insufficient_resources_nominal_columns"] :
                encode_columns_as["Hash"] = column_types["insufficient_resources_nominal_columns"]
                if encoder_name != '':
                    encoder_name = encoder_name + "_Hash"
        else:
            encode_columns_as[nominal_encoder] = nominal_cols
            encoder_name = nominal_encoder

        encoder = MetaEncoder(encode_columns_as=encode_columns_as)

        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            X_train_encoded = encoder.fit_transform(X_train, y_train)
            X_test_encoded = encoder.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Approach",
                    "dataset": "Churn",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Approach, Churn, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_churn_approach.csv", index=True)
print("Done!")

Approach, Churn, LogReg, Leave-one-out, 1, 1, 0.8115654851839926, 0.46960899433971326, 0.867, 0.3248730964467005, 0.5714285714285714, 0.22695035460992907, 0.046874046325683594
Approach, Churn, LogReg, Leave-one-out, 1, 2, 0.8319338832057728, 0.47741060885073544, 0.867, 0.3316582914572864, 0.5689655172413793, 0.23404255319148937, 0.038004159927368164
Approach, Churn, LogReg, Leave-one-out, 1, 3, 0.8283341176859123, 0.4705227008848628, 0.87, 0.3434343434343434, 0.5964912280701754, 0.24113475177304963, 0.03607630729675293
Approach, Churn, LogReg, Leave-one-out, 1, 4, 0.8403181325716538, 0.503832849614322, 0.866, 0.33, 0.5689655172413793, 0.2323943661971831, 0.03627729415893555
Approach, Churn, LogReg, Leave-one-out, 1, 5, 0.8059850947174892, 0.41428978284579576, 0.858, 0.2828282828282828, 0.5, 0.19718309859154928, 0.04204678535461426
Approach, Churn, LogReg, Leave-one-out, 2, 1, 0.8507170633839447, 0.5035926427284378, 0.876, 0.36082474226804123, 0.660377358490566, 0.24822695035460993, 0.0